This notebook is used to load and test the HRS Cohort table.  It extracts distinct HACOHORT values from the RAND longitudinal data and populate the cohort table.

**Purpose:** Load the HRS Cohort reference table.

**Source Table:** `dev_catalog.brz_raw_hrs.randhrs1992_2022v1`  
**Target Table:** `dev_catalog.slv_cdm_hrs.cohort`
**Load Script:** `sql/dml/load_hrs_cohort_data.sql`
**Validation Script:** `sql/validataion/verify_hrs_cohort_data.sql`

**Process:**
1. Clear/truncate the HRS COHORT table .
2. Extract distinct HACOHORT values from source data and load the HRS COHORT table.
3. Validate the table data.
4. Display summary stats.

In [ ]:
# -----------------------------------------------------------------------------
# Notebook Configuration
# -----------------------------------------------------------------------------

TRUNCATE_TABLE = True

TARGET_TABLE = "dev_catalog.slv_cdm_hrs.hrs_cohort"

LOAD_SQL = "dml/load_hrs_cohort_data.sql"

VALIDATION_SQL = "validation/verify_hrs_cohort_data.sql"

SOURCE_TABLE = "dev_catalog.brz_raw_hrs.randhrs1992_2022v1"

In [ ]:
# Step 1:
# Clear existing cohort data if needed (use with caution)
# Uncomment the line below to truncate the table before loading

if TRUNCATE_TABLE:
    print(f"Truncating {TARGET_TABLE}...")
    spark.sql(f"TRUNCATE TABLE {TARGET_TABLE}")
    print("✓ Target table truncated")
else:
    print("Skipping table truncation.")

In [ ]:
# Step 2
# Load distinct HRS COHORT data to the table
import sys
#sys.path.append("/Workspace/Users/peteperez.lv@gmail.com/hrs_dbx_repo/src")
#from common.sql_utils import execute_sql_file
from src.common.sql_utils import execute_sql_file

print("Loading HRS cohort data...")
execute_sql_file(
    spark,
    LOAD_SQL
)
print("✓ HRS Cohort data loaded successfully")

In [ ]:
# # Step 3: Verify the cohort table

print("Running validation queries...")

# Read and execute the SQL file (handles multiple statements)
with open("../../sql/validation/verify_hrs_cohort_data.sql", "r") as f:
    sql_content = f.read()
    # Split by semicolons and execute each statement
    statements = [s.strip() for s in sql_content.split(';') if s.strip()]
    
    for i, stmt in enumerate(statements, 1):
        print(f"\nExecuting statement {i}...")
        result_df = spark.sql(stmt)
        display(result_df)

print("\n✓ All validation queries completed")

In [ ]:
# Step 4
# Display summary statistics

source_count = spark.sql("""
    SELECT COUNT(DISTINCT HACOHORT) as distinct_cohorts
    FROM dev_catalog.brz_raw_hrs.randhrs1992_2022v1
    WHERE HACOHORT IS NOT NULL
""").collect()[0][0]

target_count = spark.sql("""
    SELECT COUNT(*) as cohort_count
    FROM dev_catalog.slv_cdm_hrs.hrs_cohort
""").collect()[0][0]

print("=" * 60)
print("HRS COHORT DATA LOAD SUMMARY")
print("=" * 60)
print(f"Distinct HACOHORT values in source: {source_count}")
print(f"Total records in HRS cohort table:      {target_count}")
print("=" * 60)

if source_count == target_count:
    print("✓ SUCCESS: All distinct HRS cohorts loaded")
else:
    print(f"⚠ WARNING: Mismatch detected. Please review.")